# Forward model: one angle of the scan, DMD vs D2NN

Everything here is computed **live** from `configs/default.py` (no cached operator), so it follows any change to the physics.

```
plane wave (alpha, theta) -> D2NN (2 layers, 0.32 um neurons, 3.3 um apart) -> 6.6 um gap (scalar AS)
  -> 4F object plane -> NA-0.8 relay aperture -> vectorial 4F relay (exPSF) into 3D -> |E|^2 illumination
  -> emission PSF (NA 0.1) + 12x12 detector  =  one column block of A
DMD: binary 0.32 um pattern in focus at the 4F object plane, same relay.
```

Single-pattern views take seconds. `show_depth_corr` / `show_column` compute all 256 patterns once per coder (~a minute), then are cached. Engine: `notebooks/fwdlab.py`.

In [ ]:
%matplotlib inline
import os, sys
p = os.getcwd()
while p != os.path.dirname(p) and not os.path.isdir(os.path.join(p, 'pipeline')):
    p = os.path.dirname(p)
os.chdir(p); sys.path[:0] = [p, os.path.join(p, 'notebooks')]
import fwdlab
lab = fwdlab.Lab()          # device='cpu' if the GPU is busy with a build

In [ ]:
dmd = lab.coder('dmd')                      # seed 0 = the patterns in the DMD operator build
rnd = lab.coder('random')                   # seed 0 = the random D2NN operator (random_gap66)
coders = [dmd, rnd]

# a trained coder: any coder_final.pt (layer count is read from the checkpoint)
import glob; print(*sorted(glob.glob('results/coder/**/coder_final.pt', recursive=True)), sep='\n')
# des = lab.coder('designed', ckpt='results/coder/physical_gap_v6s2/<run>/coder_final.pt'); coders.append(des)

## 1. The angle scan
16 azimuths alpha x 16 polar angles theta, shown in sine space. Nearby points = nearby input angles (memory-effect neighbours).

In [ ]:
lab.show_scan(idx=37)

## 2. The 4F object plane: before and after the NA-0.8 cut
The post-cut field is what the relay images into the sample. `light kept` = mean intensity per unit input (the equal-power scale). The DMD's amplitude pattern has a flat phase, which is why its 3D spread is symmetric in z.

In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider
interact(lambda idx: lab.show_plane(coders, idx), idx=IntSlider(37, 0, lab.T - 1, continuous_update=False));

## 3. The 3D illumination of one pattern
XY at chosen depths, XZ / YZ cuts (depth up). Move the slider to scan angles.

In [ ]:
interact(lambda idx, row: lab.show_volume(coders, idx, y=row),
         idx=IntSlider(37, 0, lab.T - 1, continuous_update=False),
         row=IntSlider(48, 0, lab.Nc - 1, continuous_update=False));

## 4. The +d / -d mirror for one pattern
DMD: the two planes are the same pattern (corr ~ 1), so the camera cannot tell a fluorophore at +d from one at -d. D2NN: different patterns.

In [ ]:
interact(lambda idx, d_um: lab.show_mirror(coders, idx, d_um),
         idx=IntSlider(37, 0, lab.T - 1, continuous_update=False),
         d_um=FloatSlider(value=2.56, min=0.32, max=4.8, step=0.32, continuous_update=False));

## 5. Over the whole scan: what each voxel column sees at each depth
Per pixel, correlate the 256-pattern sequence at depth a with depth b (averaged over pixels). The anti-diagonal is the mirror; bright = degenerate. First call computes all patterns per coder.

In [ ]:
lab.show_depth_corr(coders)

## 6. Columns of the operator A
The detector response of one recon voxel (0.64 um) to every pattern, next to its mirror voxel. Same construction as `build_operator.py`, so these are exactly A's columns. Column corr ~ 1 = the reconstruction cannot separate the two depths.

In [ ]:
interact(lambda z, x, y: lab.show_column(coders, z, x, y),
         z=IntSlider(3, 0, 15, continuous_update=False),
         x=IntSlider(24, 0, 47, continuous_update=False), y=IntSlider(24, 0, 47, continuous_update=False));

## 7. The D2NN phase masks

In [ ]:
for c in coders:
    lab.show_layers(c)

## 8. Play: change the physics
`Lab(cfg=...)` takes any config, e.g. another gap, NA or layer count. Coders must be re-created on the new lab.

In [ ]:
from configs.default import get_config
cfg = get_config(); cfg.d2nn_out_dist_um = 3.3
lab33 = fwdlab.Lab(cfg=cfg)
lab33.show_plane([lab33.coder('random')], idx=37)